In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from groq import Groq
api_key=os.getenv("api_key")
GROQ_MODEL= "llama-3.3-70b-versatile"
client=Groq(api_key=api_key)

import json
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings

# For Api Rate limiting
import logging
from tenacity import (
     retry,  # Decorator that wraps a function with retry logic
    stop_after_attempt,  # Stop after N total attempts
    wait_exponential,  # Wait 1s, 2s, 4s, 8s between retries
    before_sleep_log, 

)


logging.basicConfig(
    level=logging.INFO,
    filename=r"..\logs\sindhu_rag.log",
    filemode="w",

)

logger = logging.getLogger(__name__)
attempt_counter={"n":0}


# Declaring some important variable 
Embendding_Model_Name='thenlper/gte-large'
Chroma_Path=r"..\Storage\sindhu_db"
        
Top_k=3# retrieving only 3 chunks because of api costs
collection="sindh-10k-collection"
embeddings=SentenceTransformerEmbeddings(model_name=Embendding_Model_Name)



C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4980\1067917454.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4980\1067917454.py:40: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=SentenceTransformerEmbeddings(model_name=Embendding_Model_Name)
d:\Sindhu-RAG\sindhu_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See 

In [2]:
vectore_Store=Chroma(
    collection_name=collection,
    persist_directory=Chroma_Path,
    embedding_function=embeddings
)
# declaring the retreiver
retriever=vectore_Store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":Top_k}
)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4980\1769221126.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectore_Store=Chroma(


In [3]:
# function for retreiving the chunks and returing it in the format of list of dictionaries
def retrieve_chunks(user_query,retriever):
    docs=retriever.invoke(user_query)
    retreived=[]
    for i,doc in enumerate(docs):
        retreived.append({
            "Index":i,
            "Text":doc.page_content,
            "Metadata":doc.metadata
        })
    return retreived



In [4]:
def loading_bundle(path):
    with open(path,"r") as f:
        dict=json.load(f)
    return dict

path=r"..\config\v2_config.json"
bundle=loading_bundle(path)

def loading_prompt(bundle):
    system_path=bundle["prompt"]
    with open(system_path,"r") as f:
        p=f.read()
    return p



In [5]:
# --- Configuration ---
HISTORY_FILE = r"..\Storage\conversation\conversation_history.json"
MAX_MESSAGE=1


# --- Helper functions for managing conversation history ---
def load_conversation_history():
    """Loads conversation history from a JSON file, or initializes it with the system message."""
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            history = json.load(f)
      
        print(f"Loaded conversation history from {HISTORY_FILE}. Current turns: {len(history)-1}")
    if len(history)<MAX_MESSAGE:
        return history
    dropped=history[:-MAX_MESSAGE] # from top to -3 is dropped
    kept=history[-MAX_MESSAGE:] # from -3 to end is kept
    return kept
display(load_conversation_history())

Loaded conversation history from ..\Storage\conversation\conversation_history.json. Current turns: 4


[{'role': 'assistant',
  'content': 'The Indus Valley Civilization is located in the Indo-Pakistani subcontinent, with its distribution extending westward to the Makran coast and Saurashtra, and eastward into the valley of the Yamuna (Jumna).'}]

In [7]:
# Building the system Prompt
SYSTEM_MESSAGE = loading_prompt(bundle).strip()
# building the context text from the retreived chunks
def build_context_block(retrieve_chunks):
    parts=[]
    for chunk in retrieve_chunks:
        parts.append(chunk["Text"])

    return "\n\n".join(parts)
# building the user query from the by atatching the context 
def build_user_message(user_query,retrieve_chunks):
    context_text=build_context_block(retrieve_chunks)
    conversation_history=load_conversation_history()

    return f"###Question\n{user_query}\n###Conversation history\n{conversation_history}\n###context\n{context_text}"



In [9]:
# Generating the answers:

@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1,min=1,max=10),
    before_sleep=before_sleep_log(logger,logging.WARNING)
)
def generate_answer(user_message,system_message):
    response = client.chat.completions.create(
                    model=bundle["model"],
                    messages=[
                        {"role":"system","content":system_message},
                        {"role":"user","content":user_message}
                    ],
                     )

    return response.choices[0].message.content.strip()





In [11]:
# will use this function in case want to see the metadata for inquiry
def rag_answer(user_query, retriever):
    """End-to-end: retrieve → build messages → generate → return audit bundle."""
    retrieved = retrieve_chunks
    user_message = build_user_message(user_query, retrieved)
    answer = generate_answer(SYSTEM_MESSAGE)
    return {"answer": answer, "retrieved_chunks": retrieved, "user_message": user_message}


In [17]:
# --- Configuration ---
HISTORY_FILE = r"..\Storage\conversation\conversation_history.json"
MAX_MESSAGE=1


# --- Helper functions for managing conversation history ---
def load_conversation_history():
    """Loads conversation history from a JSON file, or initializes it with the system message."""
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE, 'r') as f:
            history = json.load(f)
      
        print(f"Loaded conversation history from {HISTORY_FILE}. Current turns: {len(history)-1}")
    if len(history)<MAX_MESSAGE:
        return history
    dropped=history[:-MAX_MESSAGE] # from top to -3 is dropped
    kept=history[-MAX_MESSAGE:] # from -3 to end is kept
    return kept
display(load_conversation_history())


Loaded conversation history from ..\Storage\conversation\conversation_history.json. Current turns: 4


[{'role': 'assistant',
  'content': 'Sorry, I encountered the following error: \n RetryError[<Future at 0x24cc8c80180 state=finished raised APIStatusError>]'}]

In [13]:
def save_conversation_history(history):
    """Saves the current conversation history to a JSON file."""
    with open(HISTORY_FILE, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Conversation history saved to {HISTORY_FILE}.")


In [18]:
# --- Multi-turn RAG interaction loop ---

conversation_history = load_conversation_history()
turn_count = 0
MAX_STEPS = 5
STOP_WORDS = ["exit", "quit", "stop"]

print("\n--- Starting Multi-Turn RAG Chat ---")
print("Type 'exit', 'quit', or 'stop' to end the conversation.")
print(f"Maximum {MAX_STEPS} turns allowed per session (excluding system message and previous turns).")



while turn_count < MAX_STEPS:
    user_input = input(f"\n[{turn_count + 1}/{MAX_STEPS}] You: ")

    if user_input.lower() in STOP_WORDS:
        print("Exiting chat. Goodbye!")
        break

    # 1. Retrieve relevant documents from Vector DB
    print("--> Retrieving documents for context...")
    context =retrieve_chunks(user_input,retriever)
    user_message_content = build_user_message(user_input, context)
    SYSTEM_MESSAGE = loading_prompt(bundle).strip()

    # 2. Append the user's message (with context) to history
    conversation_history.append({'role': 'user', 'content': user_message_content})

    # 3. Call the LLM with the entire history
    try:
        print("--> Calling LLM...")

        assistant_response = generate_answer(user_message_content,SYSTEM_MESSAGE)

    except Exception as e:
        assistant_response = f'Sorry, I encountered the following error: \n {e}'
        print(f"LLM Error: {e}")

    # 4. Append the LLM's response to history
    conversation_history.append({'role': 'assistant', 'content': assistant_response})

    # 5. Save the updated history
    save_conversation_history(conversation_history)

    print(f"Assistant: {assistant_response}")
    turn_count += 1

if turn_count == MAX_STEPS:
    print(f"\nMaximum turns ({MAX_STEPS}) reached. Chat session ended.")





Loaded conversation history from ..\Storage\conversation\conversation_history.json. Current turns: 4

--- Starting Multi-Turn RAG Chat ---
Type 'exit', 'quit', or 'stop' to end the conversation.
Maximum 5 turns allowed per session (excluding system message and previous turns).
--> Retrieving documents for context...
Loaded conversation history from ..\Storage\conversation\conversation_history.json. Current turns: 4
--> Calling LLM...
Conversation history saved to ..\Storage\conversation\conversation_history.json.
Assistant: The Indus Valley civilization is located in an area that stretches from Rupar to Sutkagen-dor, which is approximately 1,000 miles. Specific locations mentioned include Mohenjo-daro, Harappa, Alamgirpur, Rajasthan, Gujarat, Baluchistan, and the Indus river valley, with links to central Asia, north-eastern Afghanistan, north-eastern Persia, and south India.
--> Retrieving documents for context...
Loaded conversation history from ..\Storage\conversation\conversation_hi